## Recovers tcv kmeans results and tries out both designs
#### Design 1: Test time: Same `r` for all counties in a cluster
#### Design 2: Test time: Separate `r` for all counties but same window used to estimate them

In [1]:
import matplotlib.pyplot as plt
import pandas as pd

import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.impute import SimpleImputer

from collections import defaultdict
from multiprocessing import Pool, cpu_count
from joblib import Parallel, delayed, load
from tqdm.notebook import tqdm

import ast
import glob
import pickle
import dask
import os
import itertools

import dask.dataframe as dd
from dask.distributed import Client

from pprint import pprint

### Load the clustered panel dataset

In [2]:
hhs_clustered_panel_data = pd.read_csv("./hhs_clustered_panel_data.csv")
hhs_clustered_panel_data.head()

,fips,datetime,days_from_start,rolled_cases,log_rolled_cases,county,state,kmeans_k=2_labels,kmeans_k=3_labels,kmeans_k=4_labels,...,kmeans_k=2300_labels,kmeans_k=2400_labels,kmeans_k=2500_labels,kmeans_k=2600_labels,kmeans_k=2700_labels,kmeans_k=2800_labels,kmeans_k=2900_labels,kmeans_k=3000_labels,kmeans_k=3136_labels,kmeans_k=1_labels
0,1001.0,2020-03-30,69.0,5.142857,1.637609,Autauga,Alabama,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1001.0,2020-03-31,70.0,6.000000,1.791759,Autauga,Alabama,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1001.0,2020-04-01,71.0,6.857143,1.925291,Autauga,Alabama,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1001.0,2020-04-02,72.0,7.428571,2.005334,Autauga,Alabama,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1001.0,2020-04-03,73.0,8.285714,2.114533,Autauga,Alabama,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Load the tcv validation time window
#### Formatting of each value is: `best_windows_rmse, best_windows_mae, test_rmse, test_mae`
#### Test rmse and mae are from 1st design

In [3]:
def read_pickle(fname):
    with open(fname, "rb") as f:
        return pickle.load(f)

    
pickle_files = [os.path.join("kmeans_tcv_results", f) for f in os.listdir("kmeans_tcv_results") if f.endswith('.pickle')]
pickle_files = sorted(pickle_files)
pool = Pool()

cluster_key_list = [ast.literal_eval(s.split("=")[1].split(".")[0]) for s in pickle_files]

dfs = pool.map(read_pickle, pickle_files)

dfs_dict = dict(zip(cluster_key_list, dfs))

In [4]:
dfs_dict[(10,1)]

{(10,
  1): ({23: 2,
   24: 2,
   25: 2,
   26: 2,
   27: 2,
   28: 2,
   29: 2,
   30: 2,
   31: 2,
   32: 2,
   33: 2,
   34: 2,
   35: 2,
   36: 2,
   37: 2,
   38: 2,
   39: 2,
   40: 2,
   41: 2,
   42: 2,
   43: 2,
   44: 2,
   45: 2,
   46: 2,
   47: 2,
   48: 2,
   49: 2,
   50: 2,
   51: 2,
   52: 2,
   53: 2,
   54: 2,
   55: 2,
   56: 2,
   57: 2,
   58: 2,
   59: 2,
   60: 2,
   61: 2,
   62: 2,
   63: 2,
   64: 2,
   65: 2,
   66: 2,
   67: 2,
   68: 2,
   69: 2,
   70: 2,
   71: 2,
   72: 2,
   73: 2,
   74: 2,
   75: 2,
   76: 2,
   77: 2,
   78: 2,
   79: 2,
   80: 2,
   81: 2,
   82: 2,
   83: 2,
   84: 2,
   85: 2,
   86: 2,
   87: 2,
   88: 2,
   89: 2,
   90: 2,
   91: 2,
   92: 2,
   93: 2,
   94: 2,
   95: 2,
   96: 2,
   97: 2,
   98: 2,
   99: 2,
   100: 2,
   101: 2,
   102: 2,
   103: 2,
   104: 2,
   105: 2,
   106: 2,
   107: 2,
   108: 2,
   109: 2,
   110: 2,
   111: 2,
   112: 2,
   113: 2,
   114: 2,
   115: 2,
   116: 2,
   117: 2,
   118: 2,
   119: 2,

### Recompute Test Time with Differing Windows

In [5]:
def test_metric_worker(df, items, fips, t):
    X_cols = ["days_from_start"]
    y_col = "log_rolled_cases"
    
    best_rmse_windows = items[0]
    best_rmse_windowsize = best_rmse_windows[t]
    test_train_rmse = df[(t-best_rmse_windowsize <= df["days_from_start"]) & (df["days_from_start"] <= t)]
    test_data = df[df["days_from_start"] == t + 7]
    
    rmse_fips_df = test_train_rmse[test_train_rmse["fips"]==fips]
    test_fips_df = test_data[test_data["fips"]==fips]

    # Evaluate separately for each fips
    test_mse = 0
    nsamples_mse = 0
    lr_rmse = LinearRegression()
    if not rmse_fips_df.empty:
        try:
            lr_rmse.fit(rmse_fips_df[X_cols], rmse_fips_df[y_col])
            test_rmse_pred = lr_rmse.predict(test_fips_df[X_cols])
            test_mse = (mean_squared_error(test_rmse_pred, test_fips_df[y_col]))
            nsamples_mse = 1
            #test_mse_scores[t] += test_mse
            #test_mse_nsamples[t] += 1
        except:
            test_mse = 0
            nsamples_mse = 0
    
    best_mae_windows = items[1]
    best_mae_windowsize = best_mae_windows[t]
    test_train_mae = df[(t-best_mae_windowsize <= df["days_from_start"]) & (df["days_from_start"] <= t)]
    test_data = df[df["days_from_start"] == t + 7]
    
    mae_fips_df = test_train_mae[test_train_mae["fips"]==fips]
    test_fips_df = test_data[test_data["fips"]==fips]

    lr_mae = LinearRegression()
    test_mae = 0
    nsamples_mae = 0
    if not mae_fips_df.empty:
        try:
            lr_mae.fit(mae_fips_df[X_cols], mae_fips_df[y_col])
            test_mae_pred = lr_mae.predict(test_fips_df[X_cols])
            test_mae = mean_absolute_error(test_mae_pred, test_fips_df[y_col])
            nsamples_mae = 1
        except:
            test_mae = 0
            nsamples_mae = 0
    return np.asarray([t, test_mae, test_mse, nsamples_mse, nsamples_mae])

def test_time(dfs_dict, cluster_key, max_window=14):
    """
    Feed in the dict items and parse the first 2
    Test time train a separate r on different counties in a cluster
    
    Return the total square error and absolute error
    """
    # Retrieve best windows at each time period
    K, k = cluster_key
    items = dfs_dict[cluster_key][cluster_key]
    #print(items)
    best_rmse_windows, best_mae_windows, best_rmse_test_design1, best_mae_test_design2 = items
    # Retrive the data
    df = hhs_clustered_panel_data[hhs_clustered_panel_data["kmeans_k={}_labels".format(K)] == k]
    df = df[["fips", "datetime", "days_from_start", "rolled_cases", "log_rolled_cases", "county", "state", "kmeans_k={}_labels".format(K)]]

    starting_day = hhs_clustered_panel_data["days_from_start"].min()
    # Save per day mae and rmse
    test_mse_scores = defaultdict(int)
    test_mae_scores = defaultdict(int)
    
    test_mse_nsamples = defaultdict(int)
    test_mae_nsamples = defaultdict(int)
    
    days = sorted(list(best_mae_windows.keys()))
    fips_list = sorted(list(df["fips"].unique()))
    
    fips_t_list = itertools.product(fips_list, days)
    
    print("Setting up parallel for cluster_key={}".format(cluster_key))
    with Parallel(n_jobs=-1, backend='multiprocessing') as parallel:
        metrics_samples_tuple_arr = parallel(delayed(test_metric_worker)(df, items, fips, t) for fips, t in tqdm(fips_t_list))    
    metrics_samples_tuple_matrix = np.asmatrix(metrics_samples_tuple_arr)
    print("Compiling scores and nsamples for cluster_key={}".format(cluster_key))
    #pprint(metrics_samples_tuple_matrix)
    
    column_names = ["days_from_start", "total_mse", "total_mae", "nsamples_mse", "nsamples_mae"]
    metrics_df = pd.DataFrame(metrics_samples_tuple_matrix, columns=column_names)
    
    metrics_df = metrics_df.groupby("days_from_start").sum()
    metrics_df = metrics_df.reset_index()
    
    return metrics_df

In [ ]:
metrics_df = test_time(dfs_dict, (1,0))

Setting up parallel for cluster_key=(1, 0)


In [ ]:
metrics_df

In [ ]:
dfs_dict[(2000,0)]

In [ ]:
3.119817/3